# Agent 2 — Service Quality Monitor (Full-Service Restaurant)

> Build a timeline of service events from a floor camera and compute
> metrics that are normally impossible to measure: table touches per
> sitting, time between courses, walk-in bounce rate.

## What this notebook shows

This agent **doesn't need the VLM**. Semantic search across multiple
descriptive queries is enough to reconstruct a service timeline. We
run one search per event class, merge the hits, dedupe near-duplicate
events, and compute the metrics arithmetically.

It's the cheapest of the eight agents — `/search` is far cheaper than
`/vu/chat/completions` per call.

## Endpoints exercised

| Step | Endpoint | What it does |
|---|---|---|
| Retrieve (×N) | `POST /search` (BY_CLIP) | One search per service-event class |
| Reason | *(local arithmetic)* | Merge timeline, compute metrics |


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


## Helper — Visual Search

Same `/search` wrapper as the other notebooks. We use it many times in
this one (six event classes by default).


In [ ]:
def search(query, *, video_nos=None, unique_id="default", top_k=10,
           filtering_level="medium", search_type="BY_CLIP",
           datetime_taken=None, tag=None, camera_tag=None,
           latitude=None, longitude=None, max_retries=3):
    """Visual Search — POST /search.

    Returns the list of {videoNo, startTime, endTime, score, ...} matches.

    Filter parameters (all optional, combinable):
      • video_nos       restrict to specific videos (up to 100 ids)
      • datetime_taken  filter to videos captured at-or-after this timestamp
      • tag             filter to videos carrying this user-defined tag
      • camera_tag      filter to videos shot on this camera_model
      • latitude/longitude  GPS proximity filter (must be paired)

    Notes on the envelope: Visual Search responses are wrapped in
    {code, msg, data, success, failed}. code="0000" means OK; the actual
    hit list is in `data`. code="0001" ("network abnormal") is transient
    and we retry it.
    """
    body = {
        "search_param": query,
        "search_type": search_type,                 # "BY_CLIP" or "BY_AUDIO"
        "unique_id": unique_id,                      # namespace within your account
        "top_k": top_k,
        "filtering_level": filtering_level,          # "low"|"medium"|"high"
    }
    if video_nos:        body["video_nos"]     = list(video_nos)
    if datetime_taken:   body["datetime_taken"] = datetime_taken
    if tag:              body["tag"]            = tag
    if camera_tag:       body["camera_tag"]     = camera_tag
    if latitude is not None and longitude is not None:
        body["latitude"]  = latitude
        body["longitude"] = longitude

    for attempt in range(max_retries):
        r = requests.post(f"{VS_HOST}/search", headers=HEADERS, json=body, timeout=60)
        r.raise_for_status()
        envelope = r.json()
        code = envelope.get("code")
        if code == "0000":
            return envelope.get("data") or []
        if code == "0001" and attempt < max_retries - 1:
            # Transient. Backoff and retry.
            time.sleep(0.4 * (2 ** attempt))
            continue
        raise RuntimeError(f"/search failed: code={code} msg={envelope.get('msg')!r}")
    return []


## Step 1 — pick a floor-cam video

For this notebook we assume you already have a floor-cam clip indexed.
Substitute `VIDEO_NO` with one of yours.


In [ ]:
seed_hits = search("people in a room", top_k=1, filtering_level=None)
VIDEO_NO  = seed_hits[0]["videoNo"]
print(f"Analyzing video {VIDEO_NO!r}")


## Step 2 — one semantic search per event class

Each query is a free-text description of one type of service event.
Tweaking the wording is the most direct way to improve recall — the
search is semantic, so synonyms work, but phrasing matters.


In [ ]:
QUERIES = [
    ("server_approaches",   "server walks up to a table to greet or take an order"),
    ("food_delivery",       "server delivers a plate or tray of food to a table"),
    ("drink_refill",        "server refills a drink at a table"),
    ("check_presentation",  "server presents the bill or check to the table"),
    ("table_cleared",       "staff clears empty plates or wipes down a table"),
    ("walkin_no_greet",     "customer enters, waits, then walks out without being seated"),
]

raw_events = []
for label, q in QUERIES:
    hits = search(q, video_nos=[VIDEO_NO], top_k=10, filtering_level="low")
    for h in hits:
        raw_events.append({
            "type": label,
            "t_sec": float(h["startTime"]),
            "end_sec": float(h["endTime"]),
            "score": float(h["score"]),
        })
    print(f"  {label:<22} -> {len(hits)} hits")
print(f"\nTotal raw events: {len(raw_events)}")


## Step 3 — dedupe near-duplicates within the same event class

A 5-second handoff might match the query in three adjacent 1-second
clips. We collapse same-type events within 5 seconds of each other into
one, keeping the highest-scoring representative.


In [ ]:
from collections import defaultdict

def merge_dedupe(events, min_gap_sec=5.0):
    by_type = defaultdict(list)
    for e in events:
        by_type[e["type"]].append(e)
    merged = []
    for t, group in by_type.items():
        group.sort(key=lambda x: x["t_sec"])
        kept = []
        for e in group:
            if kept and (e["t_sec"] - kept[-1]["t_sec"]) < min_gap_sec:
                if e["score"] > kept[-1]["score"]:
                    kept[-1] = e
                continue
            kept.append(e)
        merged.extend(kept)
    merged.sort(key=lambda x: x["t_sec"])
    return merged

events = merge_dedupe(raw_events)
print(f"After dedupe: {len(events)} events\n")
for e in events[:10]:
    print(f"  t={e['t_sec']:>6.1f}s  type={e['type']:<22}  score={e['score']:.3f}")


## Step 4 — compute the service-quality metrics

With a clean timeline, the metrics fall out as one-liners:

- **food_delivery_count** — number of distinct food-delivery events.
- **avg_inter_course_sec** — mean gap between consecutive food deliveries.
- **table_touches** — count of `server_approaches` events.
- **bounce_count** — customers who entered, waited, and left ungreeted.


In [ ]:
import statistics

food_times = sorted(e["t_sec"] for e in events if e["type"] == "food_delivery")
gaps = [b - a for a, b in zip(food_times, food_times[1:])]

metrics = {
    "food_delivery_count": len(food_times),
    "avg_inter_course_sec": round(statistics.mean(gaps), 1) if gaps else None,
    "table_touches": sum(1 for e in events if e["type"] == "server_approaches"),
    "bounce_count":  sum(1 for e in events if e["type"] == "walkin_no_greet"),
}

print("Service-quality metrics for this shift:")
print(json.dumps(metrics, indent=2))


## Where to go next

- **Cross-shift comparison**: re-run for the previous day's footage and
  diff the metrics. Anomalies (e.g. inter-course time 3× the median)
  surface where the operational issue is.
- **Cross-store benchmarking**: the same code parameterizes over
  `video_nos=[...]` from multiple stores. The result is a per-store
  metrics table you can rank.
- **Add a VLM verification step** if recall is too noisy — for each
  candidate, ask the VLM "what service action is happening here?" with
  a strict JSON schema. That's the pattern used by agents 1, 3, 6, 7.
